# 03 — Channel Performance, Trends, Diminishing Returns & Campaign Analysis
**Purpose:** Analyze channel efficiency, spend vs revenue share, monthly seasonality, diminishing returns, and top/bottom campaign performance.
**Inputs:** `data/processed/clean.csv` (200,000 rows × 29 columns)
**Outputs:** Visual chart PNGs in `images/` (`03_roi_by_channel.png`, `03_spend_vs_revenue_share.png`, `03_monthly_spend_vs_revenue.png`, `03_diminishing_returns_scatter.png`, `03_top_bottom_campaigns_roi.png`, `03_cac_vs_cvr_scatter.png`, `03_seasonality_heatmap.png`)

### Business Question
*"Which marketing campaigns and channels actually work? Where should we invest more, and which campaigns should we stop?"*


### 1. Load Processed Data & Setup
We load the standardized cleaned dataset from `data/processed/clean.csv` and set global styling.

In [ ]:
# Path Resolver Helper: Ensures notebook runs seamlessly from both root and notebooks/ directories
import os
from pathlib import Path

BASE_DIR = Path('..') if Path('../data').exists() else Path('.')
DATA_RAW = BASE_DIR / 'data' / 'raw'
DATA_PROC = BASE_DIR / 'data' / 'processed'
DATA_SMPL = BASE_DIR / 'data' / 'sample'
IMAGE_DIR = BASE_DIR / 'images'
SQL_DIR = BASE_DIR / 'sql'
DASH_DIR = BASE_DIR / 'dashboard' / 'data'

for p in [DATA_RAW, DATA_PROC, DATA_SMPL, IMAGE_DIR, SQL_DIR, DASH_DIR]:
    p.mkdir(parents=True, exist_ok=True)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats

np.random.seed(42)

CLEAN_DATA_PATH = Path(str(DATA_PROC / 'clean.csv'))
IMAGE_DIR = Path('images')
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['axes.edgecolor'] = '#CCCCCC'

df = pd.read_csv(CLEAN_DATA_PATH)
df['date'] = pd.to_datetime(df['date'])
df['year_month'] = df['date'].dt.to_period('M').astype(str)
df['month'] = df['date'].dt.month
df['campaign_name'] = df['channel'] + " - " + df['campaign_type']

print(f"Loaded clean dataset: {df.shape[0]:,} rows, {df.shape[1]} columns.")


## Part A: Channel Comparison & Performance Ranking

### Methodology & Flagging Rules
We aggregate channel total spend, revenue, impressions, clicks, and conversions first, then calculate ratios from sums.

**Explicit Classification Rules & Thresholds:**
1. **Overall Baseline Portfolio ROI:** `+6.21%`
2. **Average Channel Spend Share:** `16.67%` (100% / 6 channels)

- **⚠️ High Volume, Poor Return:** `Spend Share > 16.67%` AND `ROI < +6.21%`
- **⭐ Small but Excellent:** `Spend Share < 16.67%` AND `ROI > +6.21%`
- **🚀 High Volume, High Return:** `Spend Share >= 16.67%` AND `ROI > +6.21%`
- **📉 Small but Poor:** `Spend Share < 16.67%` AND `ROI < +6.21%`


In [ ]:
chan_summary = df.groupby('channel').agg({
    'campaign_id': 'count',
    'impressions': 'sum',
    'clicks': 'sum',
    'conversions': 'sum',
    'spend': 'sum',
    'revenue': 'sum'
}).rename(columns={'campaign_id': 'n_campaigns'}).reset_index()

tot_spend = chan_summary['spend'].sum()
tot_rev = chan_summary['revenue'].sum()
overall_roi_pct = ((tot_rev - tot_spend) / tot_spend) * 100

chan_summary['spend_share_pct'] = (chan_summary['spend'] / tot_spend) * 100
chan_summary['rev_share_pct'] = (chan_summary['revenue'] / tot_rev) * 100
chan_summary['roi'] = (chan_summary['revenue'] - chan_summary['spend']) / chan_summary['spend']
chan_summary['roi_pct'] = chan_summary['roi'] * 100
chan_summary['cvr_pct'] = (chan_summary['conversions'] / chan_summary['clicks']) * 100
chan_summary['cac'] = chan_summary['spend'] / (chan_summary['conversions'] * 0.60)
chan_summary['effective_aov'] = chan_summary['revenue'] / chan_summary['conversions']

chan_summary['roi_rank'] = chan_summary['roi_pct'].rank(ascending=False, method='min').astype(int)
chan_summary['cac_rank'] = chan_summary['cac'].rank(ascending=True, method='min').astype(int)
chan_summary['cvr_rank'] = chan_summary['cvr_pct'].rank(ascending=False, method='min').astype(int)

avg_spend_share = 100.0 / len(chan_summary)

def flag_channel(row):
    high_vol = row['spend_share_pct'] >= avg_spend_share
    good_ret = row['roi_pct'] >= overall_roi_pct
    if high_vol and not good_ret:
        return 'High volume, poor return'
    elif not high_vol and good_ret:
        return 'Small but excellent'
    elif high_vol and good_ret:
        return 'High volume, high return'
    else:
        return 'Small but poor'

chan_summary['performance_flag'] = chan_summary.apply(flag_channel, axis=1)
chan_summary = chan_summary.sort_values('roi_rank')

display_cols = ['roi_rank', 'channel', 'spend', 'revenue', 'roi_pct', 'cac', 'cvr_pct', 
                'spend_share_pct', 'rev_share_pct', 'performance_flag']

print(f"Overall Portfolio ROI: {overall_roi_pct:.2f}% | Avg Spend Share: {avg_spend_share:.2f}%\n")
print(chan_summary[display_cols].to_string(index=False))


### 2. Headline Visual: Spend Share vs. Revenue Share
Grouped bar chart comparing capital allocation against revenue generation across channels.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(chan_summary))
width = 0.35

rects1 = ax.bar(x - width/2, chan_summary['spend_share_pct'], width, label='Spend Share (%)', color='#34495E', alpha=0.85)
rects2 = ax.bar(x + width/2, chan_summary['rev_share_pct'], width, label='Revenue Share (%)', color='#2ECC71', alpha=0.85)

ax.set_ylabel('Percentage (%)', fontsize=11, fontweight='bold')
ax.set_title('Headline Chart: Spend Share vs. Revenue Share per Channel', fontsize=13, fontweight='bold', pad=12)
ax.set_xticks(x)
ax.set_xticklabels(chan_summary['channel'], fontweight='bold')
ax.axhline(avg_spend_share, color='#888888', linestyle=':', label=f'Equal Share Baseline ({avg_spend_share:.1f}%)')
ax.legend(frameon=True, facecolor='white')

for r in rects1:
    ax.annotate(f"{r.get_height():.1f}%", (r.get_x() + r.get_width()/2, r.get_height()), xytext=(0, 3), textcoords='offset points', ha='center', fontsize=9)
for r in rects2:
    ax.annotate(f"{r.get_height():.1f}%", (r.get_x() + r.get_width()/2, r.get_height()), xytext=(0, 3), textcoords='offset points', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig(IMAGE_DIR / '03_spend_vs_revenue_share.png', dpi=150)
plt.show()


### 3. Visualizing Channel ROI %
Sorted bar chart showing positive ROI channels in green and negative ROI channels in red.

In [ ]:
plt.figure(figsize=(9, 5))
colors = ['#2ECC71' if val >= 0 else '#E74C3C' for val in chan_summary['roi_pct']]
bars = plt.bar(chan_summary['channel'], chan_summary['roi_pct'], color=colors, edgecolor='black', linewidth=0.5)

plt.axhline(0, color='black', linewidth=1)
plt.axhline(overall_roi_pct, color='#34495E', linestyle='--', label=f'Overall ROI ({overall_roi_pct:.2f}%)')

plt.ylabel('ROI (%)', fontsize=11, fontweight='bold')
plt.title('ROI (%) by Marketing Channel (Sorted)', fontsize=13, fontweight='bold', pad=12)
plt.legend(frameon=True, facecolor='white')

for bar in bars:
    height = bar.get_height()
    va = 'bottom' if height >= 0 else 'top'
    y_pos = height + (1.5 if height >= 0 else -3.5)
    plt.annotate(f"{height:+.1f}%", (bar.get_x() + bar.get_width()/2, y_pos), ha='center', va=va, fontweight='bold', fontsize=10)

plt.tight_layout()
plt.savefig(IMAGE_DIR / '03_roi_by_channel.png', dpi=150)
plt.show()


### 4. Acquisition Efficiency: CAC vs. CVR Bubble Plot
Scatter bubble plot mapping CAC against CVR, with bubble size proportional to spend.

In [ ]:
plt.figure(figsize=(9, 5.5))
# Colors mapped via case-insensitive lookup
scatter_palette = {'email': '#2E86C1', 'facebook': '#E74C3C', 'google ads': '#27AE60',
                   'instagram': '#8E44AD', 'website': '#D35400', 'youtube': '#C0392B'}

for _, row in chan_summary.iterrows():
    ch = row['channel']
    color = scatter_palette.get(ch.lower(), '#7F8C8D')
    plt.scatter(row['cvr_pct'], row['cac'], s=row['spend']/5e5, color=color, alpha=0.6, edgecolors='black', linewidth=1.5, label=ch)
    plt.annotate(f"{ch}\n(${row['cac']:.0f}, {row['cvr_pct']:.2f}%)", (row['cvr_pct'], row['cac']), xytext=(0, 10), textcoords='offset points', ha='center', fontsize=9, fontweight='bold')

plt.xlabel('Conversion Rate - CVR (%)', fontsize=11, fontweight='bold')
plt.ylabel('Customer Acquisition Cost - CAC ($)', fontsize=11, fontweight='bold')
plt.title('CAC vs. CVR per Channel (Bubble Size = Total Spend)', fontsize=13, fontweight='bold', pad=12)
plt.legend(title='Channel', fontsize=9)
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.savefig(str(IMAGE_DIR / '03_cac_vs_cvr_scatter.png'), dpi=150)
plt.show()


## 5. Where Metrics Mislead (Empirical Data Analysis)

1. **The CAC and CVR Illusion:** All 6 channels have virtually identical CAC ($469.88 to $475.37) and CVR (7.98% to 8.03%). Evaluating channels solely on acquisition cost misses net ROI, which ranges from **+46.79% (Instagram)** down to **-43.41% (YouTube)**.
2. **Average Order Value (AOV) Monetization:** YouTube yields an effective AOV of **$161.15**, whereas Instagram yields **$418.55**. Low CAC is meaningless without monetized output size.


### Part A Key Takeaways
1. Reallocate **$827.4M** away from YouTube (-43.41% ROI) and Facebook (-22.76% ROI), which destroyed **$268.4M** in capital.
2. Instagram is the top growth driver with **+46.79% ROI** ($195.2M net profit).
3. Google Ads delivers strong search intent returns with **+34.73% ROI** ($145.5M net profit).


---
# Part B: Trends, Seasonality & Diminishing Returns Analysis


### 1. Monthly Total Spend vs. Revenue & ROI Trend Line Chart

In [ ]:
monthly = df.groupby('year_month').agg({'spend': 'sum', 'revenue': 'sum'}).reset_index()
monthly['spend_m'] = monthly['spend'] / 1e6
monthly['revenue_m'] = monthly['revenue'] / 1e6
monthly['roi_pct'] = ((monthly['revenue'] - monthly['spend']) / monthly['spend']) * 100

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True, gridspec_kw={'height_ratios': [2.5, 1.5]})

ax1.plot(monthly['year_month'], monthly['revenue_m'], marker='o', color='#2ECC71', linewidth=2.5, label='Total Revenue ($M)')
ax1.plot(monthly['year_month'], monthly['spend_m'], marker='s', color='#E74C3C', linewidth=2.5, linestyle='--', label='Total Spend ($M)')
ax1.set_ylabel('Amount ($ Millions)', fontsize=11, fontweight='bold')
ax1.set_title('Monthly Total Marketing Spend vs. Revenue (2021)', fontsize=13, fontweight='bold', pad=12)
ax1.legend(frameon=True, facecolor='white', loc='upper left')
ax1.grid(True, linestyle=':', alpha=0.6)

ax2.bar(monthly['year_month'], monthly['roi_pct'], color='#3498DB', alpha=0.85, width=0.5)
ax2.axhline(monthly['roi_pct'].mean(), color='#2C3E50', linestyle=':', label=f"Avg Monthly ROI: +{monthly['roi_pct'].mean():.2f}%")
ax2.set_ylabel('ROI (%)', fontsize=11, fontweight='bold')
ax2.set_xlabel('Month', fontsize=11, fontweight='bold')
ax2.set_title('Overall Monthly ROI (%)', fontsize=11, fontweight='bold', pad=8)
ax2.set_ylim(0, 10)
ax2.legend(frameon=True, facecolor='white', loc='upper left')
ax2.grid(True, linestyle=':', alpha=0.6)

plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(IMAGE_DIR / '03_monthly_spend_vs_revenue.png', dpi=150)
plt.show()


### 2. Seasonality & Simulation Artifact Analysis
Uniform month-to-month ROI performance across channels is an **artifact of synthetic dataset simulation**, rather than real consumer holiday demand surges.

In [ ]:
channel_month = df.groupby(['channel', 'month']).agg({'spend': 'sum', 'revenue': 'sum'}).reset_index()
channel_month['roi_pct'] = ((channel_month['revenue'] - channel_month['spend']) / channel_month['spend']) * 100
pivot_roi = channel_month.pivot(index='channel', columns='month', values='roi_pct')
pivot_roi.columns = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

plt.figure(figsize=(11, 5.5))
sns.heatmap(pivot_roi, annot=True, fmt='.1f', cmap='RdYlGn', center=0, cbar_kws={'label': 'ROI (%)'}, linewidths=0.5, annot_kws={'size': 9, 'weight': 'bold'})
plt.title('Channel ROI (%) by Month of Year (Seasonality Check)', fontsize=13, fontweight='bold', pad=12)
plt.xlabel('Month', fontsize=11, fontweight='bold')
plt.ylabel('Channel', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(IMAGE_DIR / '03_seasonality_heatmap.png', dpi=150)
plt.show()


### 3. Log-Log Diminishing Returns & Spend Elasticity ($eta$)

In [ ]:
chan_monthly = df.groupby(['channel', 'year_month']).agg({'spend': 'sum', 'revenue': 'sum'}).reset_index()

results = []
for ch in sorted(df['channel'].unique()):
    sub = chan_monthly[chan_monthly['channel'] == ch].copy()
    
    log_spend = np.log(sub['spend'])
    log_rev = np.log(sub['revenue'])
    slope, intercept, r_val, p_val, std_err = stats.linregress(log_spend, log_rev)
    elasticity = slope
    r2 = r_val ** 2
    
    q25 = sub['spend'].quantile(0.25)
    q75 = sub['spend'].quantile(0.75)
    bot_q = sub[sub['spend'] <= q25]
    top_q = sub[sub['spend'] >= q75]
    
    bot_mroas = bot_q['revenue'].sum() / bot_q['spend'].sum()
    top_mroas = top_q['revenue'].sum() / top_q['spend'].sum()
    roas_delta = top_mroas - bot_mroas
    tot_roi = ((sub['revenue'].sum() - sub['spend'].sum()) / sub['spend'].sum()) * 100
    
    if tot_roi < 0:
        recommendation = 'saturated'
    elif elasticity < 0.85:
        recommendation = 'hold'
    else:
        recommendation = 'scale'
        
    results.append({
        'channel': ch,
        'elasticity_beta': round(elasticity, 3),
        'r2_score': round(r2, 3),
        'q1_bot_roas': round(bot_mroas, 3),
        'q4_top_roas': round(top_mroas, 3),
        'roas_delta': round(roas_delta, 3),
        'overall_roi_pct': round(tot_roi, 2),
        'decision': recommendation
    })

diminishing_df = pd.DataFrame(results)
print(diminishing_df.to_string(index=False))


### 4. Visualizing Diminishing Returns per Channel

In [ ]:
channels = sorted(df['channel'].unique())
fig, axes = plt.subplots(2, 3, figsize=(14, 8), sharex=False, sharey=False)
axes = axes.flatten()

colors = {ch: plt.cm.tab10(i) for i, ch in enumerate(sorted(df['channel'].unique()))}

for idx, ch in enumerate(channels):
    ax = axes[idx]
    sub = chan_monthly[chan_monthly['channel'] == ch].copy()
    
    x_spend = sub['spend'] / 1e6
    y_rev = sub['revenue'] / 1e6
    
    log_x = np.log(sub['spend'])
    log_y = np.log(sub['revenue'])
    slope, intercept, r_val, p_val, std_err = stats.linregress(log_x, log_y)
    
    x_fit = np.linspace(x_spend.min(), x_spend.max(), 100)
    y_fit = np.exp(intercept) * ((x_fit * 1e6) ** slope) / 1e6
    
    ax.scatter(x_spend, y_rev, color=colors[ch], alpha=0.7, edgecolors='k', s=45, label='Monthly Data')
    ax.plot(x_fit, y_fit, color='#2C3E50', linestyle='--', linewidth=2, label=f'Power Fit (β={slope:.2f})')
    
    ax.set_title(f"{ch}\nβ = {slope:.2f} | R² = {r_val**2:.2f}", fontsize=11, fontweight='bold', color=colors[ch])
    ax.set_xlabel('Spend ($M)', fontsize=9)
    ax.set_ylabel('Revenue ($M)', fontsize=9)
    ax.legend(fontsize=8, loc='upper left')
    ax.grid(True, linestyle=':', alpha=0.6)

plt.suptitle('Diminishing Returns Analysis: Log-Log Spend vs. Revenue by Channel', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(IMAGE_DIR / '03_diminishing_returns_scatter.png', dpi=150)
plt.show()


---
# Part C: Campaign Level Performance & Budget Optimization

In Part C, we drill down into campaign entities (`Channel - CampaignType`) to identify high-performing winners, expose capital-losing campaigns, and formulate actionable budget pause/reallocation recommendations.


### 1. Minimum-Spend Filtering Rationale
To prevent tiny, low-volume campaigns from distorting performance rankings due to sample noise, we apply a **25th percentile spend filter threshold** ($82.62M).

- **Total Granular Campaigns:** 30
- **Spend 25th Percentile Threshold:** `$82,617,359.50`
- **Retained Filtered Campaigns (>= P25):** 22
- **Excluded Tiny Campaigns (< P25):** 8


In [ ]:
camp_summary = df.groupby(['campaign_name', 'channel', 'campaign_type']).agg(
    n_rows=('campaign_id', 'count'),
    impressions=('impressions', 'sum'),
    clicks=('clicks', 'sum'),
    conversions=('conversions', 'sum'),
    spend=('spend', 'sum'),
    revenue=('revenue', 'sum')
).reset_index()

camp_summary['roi'] = (camp_summary['revenue'] - camp_summary['spend']) / camp_summary['spend']
camp_summary['roi_pct'] = camp_summary['roi'] * 100
camp_summary['roas'] = camp_summary['revenue'] / camp_summary['spend']
camp_summary['cvr_pct'] = (camp_summary['conversions'] / camp_summary['clicks']) * 100
camp_summary['cac'] = camp_summary['spend'] / (camp_summary['conversions'] * 0.60)
camp_summary['net_profit'] = camp_summary['revenue'] - camp_summary['spend']

p25_spend = camp_summary['spend'].quantile(0.25)
filtered_camps = camp_summary[camp_summary['spend'] >= p25_spend].sort_values('roi_pct', ascending=False).copy()
excluded_camps = camp_summary[camp_summary['spend'] < p25_spend].sort_values('spend', ascending=True).copy()

print(f"Spend Threshold (P25): ${p25_spend:,.2f}")
print(f"Filtered Campaigns: {len(filtered_camps)} | Excluded Campaigns: {len(excluded_camps)}")


### 2. Filter Effect: Unfiltered vs. Filtered Rankings Comparison
Filtering removes low-volume campaign cells that distort the bottom-tier list. For instance, `YouTube - Email` (spend $82.24M, ROI -44.08%) was excluded by the spend filter, allowing `Website - Search` (spend $85.65M, ROI +2.26%) to appear in the filtered bottom 5 ranking table.

In [ ]:
unfiltered_top5 = camp_summary.sort_values('roi_pct', ascending=False).head(5)['campaign_name'].tolist()
filtered_top5 = filtered_camps.head(5)['campaign_name'].tolist()
unfiltered_bot5 = camp_summary.sort_values('roi_pct', ascending=True).head(5)['campaign_name'].tolist()
filtered_bot5 = filtered_camps.sort_values('roi_pct', ascending=True).head(5)['campaign_name'].tolist()

comp_df = pd.DataFrame({
    'Unfiltered Top 5': unfiltered_top5,
    'Filtered Top 5': filtered_top5,
    'Unfiltered Bottom 5': unfiltered_bot5,
    'Filtered Bottom 5': filtered_bot5
})
print(comp_df.to_string(index=False))


### 3. Top 5 and Bottom 5 Filtered Campaigns Table

In [ ]:
top5_df = filtered_camps.head(5).copy()
bot5_df = filtered_camps.sort_values('roi_pct', ascending=True).head(5).copy()

top_bot_df = pd.concat([top5_df, bot5_df])
disp_cols = ['campaign_name', 'channel', 'spend', 'revenue', 'roi_pct', 'roas', 'cac', 'cvr_pct']

print("--- Top 5 and Bottom 5 Filtered Campaigns ---")
print(top_bot_df[disp_cols].to_string(index=False))


### 4. Negative ROI Campaigns & Total Money Lost Audit
Out of 30 campaigns, exactly **10 campaigns generated negative ROI** (all 5 YouTube campaigns and all 5 Facebook campaigns). Total capital lost across these 10 campaigns equals **$274,364,945.35**.

In [ ]:
neg_camps = camp_summary[camp_summary['roi'] < 0].sort_values('net_profit', ascending=True).copy()
tot_neg_loss = abs(neg_camps['net_profit'].sum())

print(f"Total Negative ROI Campaigns: {len(neg_camps)} / {len(camp_summary)}")
print(f"Total Net Capital Loss: ${tot_neg_loss:,.2f}\n")
print(neg_camps[['campaign_name', 'channel', 'spend', 'revenue', 'net_profit', 'roi_pct']].to_string(index=False))


### 5. Visualizing Top 5 vs. Bottom 5 Campaign ROI
Horizontal bar chart comparing the top 5 positive ROI campaigns against the bottom 5 campaigns.

In [ ]:
top5 = filtered_camps.head(5)
bot5 = filtered_camps.sort_values('roi_pct', ascending=True).head(5)
plot_df = pd.concat([bot5, top5])

plt.figure(figsize=(10, 6))
colors = ['#E74C3C' if val < 0 else '#2ECC71' for val in plot_df['roi_pct']]
bars = plt.barh(plot_df['campaign_name'], plot_df['roi_pct'], color=colors, edgecolor='black', linewidth=0.5)

plt.axvline(0, color='black', linewidth=1)
plt.xlabel('ROI (%)', fontsize=11, fontweight='bold')
plt.title('Top 5 and Bottom 5 Campaigns by ROI (Filtered >= P25 Spend)', fontsize=13, fontweight='bold', pad=12)

for bar in bars:
    width = bar.get_width()
    ha = 'left' if width >= 0 else 'right'
    x_pos = width + (1.5 if width >= 0 else -1.5)
    plt.annotate(f"{width:+.1f}%", (x_pos, bar.get_y() + bar.get_height()/2), va='center', ha=ha, fontweight='bold', fontsize=9)

plt.xlim(-55, 60)
plt.tight_layout()
plt.savefig(IMAGE_DIR / '03_top_bottom_campaigns_roi.png', dpi=150)
plt.show()


### 6. Actionable Campaign Pause Recommendations

Based on empirical performance data, we recommend pausing or restructuring the following campaign initiatives:

1. **⛔ Pause `YouTube - Search` & `YouTube - Influencer`**
   - **Data:** Combined Spend = **$168.1M**, Revenue = **$94.7M**, Net Loss = **$73.46M** (ROI $\approx -43.7\%$).
   - **Cautionary Note:** Prior to complete deactivation, conduct an Assisted Conversions & Multi-Touch Attribution (MTA) audit. YouTube search/influencer video content may drive top-of-funnel brand awareness that assists downstream direct/organic sales.
2. **⛔ Pause `Facebook - Influencer` & `Facebook - Email`**
   - **Data:** Combined Spend = **$164.4M**, Revenue = **$125.4M**, Net Loss = **$38.99M** (ROI $\approx -23.7\%$).
   - **Cautionary Note:** Verify pixel tracking setups and audience overlap with Instagram before pausing. Retargeting segments on Facebook may be capturing warm leads that convert on other channels.
3. **⏸️ Restructure / Pause `Website - Search`**
   - **Data:** Spend = **$85.65M**, Revenue = **$87.58M**, Net Profit = **$1.94M** (ROI = **+2.26%**).
   - **Cautionary Note:** Website search includes brand keyword bidding. Halting brand search campaigns might allow competitors to capture high-intent branded traffic.


### Phase 6 Final Executive Summary & Next Steps

1. **Instagram Domination:** Instagram campaigns capture all 5 top spots (ROI +45.6% to +48.8%), proving to be our premier acquisition vehicle.
2. **Eliminate Capital Loss:** Deactivating or scaling down YouTube and Facebook campaigns frees up **$827.4M in budget**, halting **$274.4M in net capital destruction**.
3. **Reallocation Target:** Reallocate 70% of freed capital into Instagram and Google Ads, and 30% into conversion rate optimization (CRO) for Website landing pages.
